# BSHI construction and validation

Builds the window index, seven-plane beat images, masks, and the development subset. Includes input-range and throughput checks.

Run the setup cell first. Full processing and training require locally provisioned datasets; the grading quick start uses synthetic tests and saved aggregate results.


In [ ]:
from pathlib import Path
import sys, json, os, time, shutil
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import importlib, psutil
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from hemodynamics import snapshot, case_quality as CQ, bshi as S, bshi_checks as K, bshi_bench as B, labels as L
for _module in (snapshot, CQ, S, K, B):   # pick up edited code in a long-lived kernel
    importlib.reload(_module)

me = psutil.Process()
for p in psutil.process_iter():
    try: p.cpu_percent(None)
    except psutil.Error: pass
time.sleep(0.8)
busy = []
for p in psutil.process_iter(['pid', 'name']):
    try:
        if p.pid != me.pid and p.info['name'] and 'python' in p.info['name'].lower() and p.cpu_percent(None) > 50:
            busy.append(p.pid)
    except psutil.Error:
        pass
print('Root:', ROOT)
print(f'Free disk: {shutil.disk_usage(ROOT).free / 1e9:.0f} GB | free RAM: {psutil.virtual_memory().available / 1e9:.0f} GB | CPUs: {os.cpu_count()}')
print('Other busy python processes:', busy or 'none')
print('Dataset tag:', snapshot.assert_dataset_v1(ROOT)['version'], '(the build refuses to run if the dataset changed)')
if busy:
    print('WARNING: another job is using the CPU; the throughput benchmark in section 4 would be misleading. Wait for it first.')

## 1. Case-quality flags and the dead-ECG decision
`ecg_dead` = ECG SQI < 0.5, **and** the arterial pulse cannot be tied to the ECG beats (`art_ok` < 10%), **and** the ECG heart rate contradicts the independently measured arterial pulse rate (> 25 bpm) or the raw ECG amplitude is < 25% of the dataset median. The first draft (SQI < 0.3, `assoc_failed` > 90%) recovered 61 of the 68 and flagged 2 extra because the `assoc_failed` flag bit is shared with PPG association failures; the rule above recovers exactly the 68. Thresholds are in `case_quality.RULE`.

In [ ]:
FORCE = False   # True recomputes everything below even if the inputs are unchanged
quality = CQ.build(ROOT, progress=True, force=FORCE)          # RUN 1a - START HERE (about 2 min the first time, instant afterwards)
decision = CQ.build_dead_ecg_decision(ROOT)
ev, dec = decision['evidence'], decision['decision']
display(quality.groupby('dataset')[['ecg_dead', 'art_dead', 'ppg_dead', 'ppg_suspect_case', 'bshi_core_possible', 'bshi_full_possible']].sum())
display(pd.Series({
    'dead-ECG cases (VitalDB) / reference set': f"{ev['flagged_by_dataset']} / {ev['reference_set_art_ok_frac_below_0.1_vitaldb']}", 'recovered / missed / extra': f"{ev['recovered']} / {len(ev['missed_from_reference'])} / {len(ev['extra_beyond_reference'])}",
    'kinds': ev['flagged_by_kind'], 'patients (own other healthy cases)': f"{ev['patients']} ({ev['patients_with_other_healthy_cases']})", 'split distribution': ev['split_distribution'],
    'ready hours dead / rest': f"{ev['ready_hours_dead']} / {ev['ready_hours_rest']}", 'eligible windows / events (dead)': f"{ev['primary_windows_and_events_dead']['windows']} / {ev['primary_windows_and_events_dead']['events']}",
    'event prevalence dead vs rest': f"{100 * ev['map_features_dead']['event_prevalence']:.2f}% vs {100 * ev['map_features_rest']['event_prevalence']:.2f}%",
    'MAP-only AUROC on dead cases': f"{ev['baseline_A_style_auroc']['dead_cases_all_splits']['auroc']:.3f} {ev['baseline_A_style_auroc']['dead_cases_all_splits']['auroc_ci95']}",
    'MAP-only AUROC on the rest (test)': f"{ev['baseline_A_style_auroc']['rest_test']['auroc']:.3f} {ev['baseline_A_style_auroc']['rest_test']['auroc_ci95']}", 'checks': dec['checks']}, name='dead-ECG evidence').to_frame())
print('DECISION:', dec['decision'])
print('Rejected alternative:', dec['rejected_alternative'])

## 2. Window index
One row per labeled window (1.41 M): the 60 most recent beats whose whole cycle ends by the forecast origin, per-window counts of usable beats, and the flags `primary_eligible`, `in_core` (>= 54 of 60 beats with a valid ART pulse) and `in_full` (also usable PPG with an ECG->PPG delay <= 0.9 s). Resumable by case (per-case parts with their own signature).

In [ ]:
res = S.build_window_index(ROOT, progress=True, force=FORCE)  # RUN 2 - START HERE (about 1 min the first time)
print(res)
summ = S.index_summary(ROOT)
f = pd.DataFrame(summ['funnel']['split_pooled'])
display(f[(f.split == 'all')].pivot(index='step', columns='dataset', values='windows').loc[['all_labeled', 'primary_eligible', 'primary_has60', 'primary_core', 'primary_full']])
cell = f[(f.dataset != 'all') & (f.split != 'all') & f.step.isin(['primary_core', 'primary_full'])]
display(cell.pivot_table(index=['step', 'dataset'], columns='split', values=['windows', 'events', 'patients_with_event']))
print('removed by the dead-ECG rule:', summ['removed_by_dead_ecg_rule'], '| eligible windows without 60 beats:', summ['windows_without_60_beats_among_primary'])
for k, v in summ['reconciliation'].items():
    print(k, 'now', v['now'], '| earlier ad-hoc anchor', v['earlier_adhoc_anchor']['windows'], 'windows ->', v['difference_pct_windows'], '%')
print('(The small difference to the earlier ad-hoc count is the rule that a beat must END by t0: in MOVER the in-progress cycle is often a gap-spanning invalid beat.)')

## 3. Tensor checks on real windows and a gallery
2,000 random full-set windows (both datasets, all splits) and 500 core-only windows are assembled and checked: no NaN or Inf, no constant plane, ECG/PPG within the +-8 clip, the PPG plane exactly zero in `ea`. Per-dataset plane statistics are the dataset-shift evidence. The gallery shows 8 real windows (2 events and 2 controls per dataset), EAP on the top row and EA below: ECG, ART, PPG, MAP z-score, timing bands, morphology bands, SQI bands.

In [ ]:
checks = K.run_tensor_checks(ROOT)                           # RUN 3 - START HERE (about 40 s)
print('checks:', checks['checks_pass'])
rows = []
for name, v in checks['checks'].items():
    for ds, d in v['per_dataset'].items():
        rows.append({'set': name, 'dataset': ds, 'windows': d['windows'], **{f'{p} mean/std': f"{s['mean']:.2f}/{s['std']:.2f}" for p, s in d['planes'].items()}, **{f'masked {k}': d['masked_beat_share'][k] for k in d['masked_beat_share']}})
display(pd.DataFrame(rows))
paths = K.draw_gallery(ROOT)
for p in paths[:4]:
    print(p); display(Image(filename=str(ROOT / p)))
print(f'{len(paths)} gallery images are in figures/bshi/ (the first 4 are shown here)')

## 4. Loader throughput
Case-grouped batches with an LRU case cache, measured as windows per second: one process, 8 and 16 worker processes, and a real torch `DataLoader` (16 persistent workers; starting 16 workers takes about a minute on Windows, so epoch 1 is slow and epoch 2 shows the steady state). Target: >= 3,000 windows/s. If it were below 1,500 at 16 workers a memory-mappable per-case store would be built; it is not needed when the target is met. Set `RUN_BENCHMARK = False` to only show the saved result. Do not run anything else heavy meanwhile.

In [ ]:
RUN_BENCHMARK = True
if RUN_BENCHMARK:
    bench = B.benchmark_throughput(ROOT)                      # RUN 4 - START HERE (about 2 min)
else:
    bench = json.loads((ROOT / 'results' / 'bshi_throughput.json').read_text())
display(pd.DataFrame(bench['assemble_only']).T)
print('torch DataLoader:', json.dumps(bench.get('torch_dataloader'), indent=1))
print('target met at 16 workers:', bench['target_met_at_16_workers'], '|', bench['decision'])

## 5. Dev subset
A small deterministic, patient-diverse sample of full-set windows (EAP, compact binary parquet: image planes 1-3 as float16 bytes, the 60 x 17 per-beat scalar matrix as float32 bytes, masks, context, labels, `sampling_weight`). At most 12 windows per case, MOVER share >= 20%, event share >= 25%, <= 3 GB. The size is limited by the MOVER supply under the per-case cap. `sampling_weight` recovers natural prevalence in weighted metrics. This is the data for the smoke test and the PySpark setup.

In [ ]:
dev = B.build_dev_subset(ROOT, force=FORCE)                  # RUN 5 - START HERE (about 1 min the first time)
display(pd.DataFrame(dev['per_split']).T)
print({k: dev[k] for k in ('gb', 'windows', 'parts', 'patients', 'cases', 'max_windows_per_case')})
print('format:', dev['format'])
print('round-trip check:', B.verify_dev_subset(ROOT))

## 6. Verification
All lines should print `True`. If anything is `False`, send the printed lines back.

In [ ]:
q = CQ.load_case_quality(ROOT)
idx = S.load_window_index(ROOT)
meta = json.loads(S._meta_path(ROOT).read_text())
dev = json.loads((ROOT / 'results' / 'bshi_dev_subset.json').read_text())
dd = json.loads((ROOT / 'results' / 'dead_ecg_decision.json').read_text())
tc = json.loads((ROOT / 'results' / 'bshi_tensor_checks.json').read_text())
th = json.loads((ROOT / 'results' / 'bshi_throughput.json').read_text())
cfgd = json.loads((ROOT / 'configs' / 'hemodynamics.json').read_text())['bshi']['dev_subset']
verify = B.verify_dev_subset(ROOT)
checks = {
    'dataset v1 guard': snapshot.assert_dataset_v1(ROOT)['version'] == 'v1',
    'the 68 dead-ECG VitalDB cases are flagged and recovered exactly': dd['evidence']['recovered'] == 68 and int(q.ecg_dead.sum()) == 68 and dd['decision']['default_kept'],
    'no dead-ECG case is in any BSHI set': int((idx.in_core & idx.ecg_dead_case).sum()) == 0 and int((idx.in_full & idx.ecg_dead_case).sum()) == 0,
    'the window index covers every labeled window': len(idx) == len(L.load_labels(ROOT, columns=['case_id'])),
    'window index signature is current': meta['signature'] == S.index_signature(ROOT),
    'index integrity (no patient in two splits, full inside core, 60 beats per image)': bool(S.check_index(idx)),
    'tensor checks pass on real windows': all(tc['checks_pass'].values()),
    'dev subset: size, per-case cap, MOVER share, event share': dev['gb'] <= cfgd['max_gb'] and dev['max_windows_per_case'] <= cfgd['max_per_case']
        and all(v['mover_share'] >= cfgd['min_mover_share'] and v['event_share'] >= cfgd['min_event_share'] for v in dev['per_split'].values()),
    'dev subset decodes to the same tensors as a fresh assembly': verify['planes_3_6_exactly_equal'] and verify['context_equal'] and verify['labels_equal'] and verify['planes_0_2_max_abs_diff'] < 0.01,
    'loader throughput target met at 16 workers': th['target_met_at_16_workers'],
}
for k, v in checks.items(): print(f'{k}: {v}')
print('ALL CHECKS PASSED' if all(checks.values()) else 'SOMETHING IS FALSE: send the lines above back')

## 7. Training readiness
Is the BSHI output sufficient to train the networks? The cell below runs `hemodynamics.bshi_audit` (about 4 minutes: it assembles 3,000 windows for the input checks, about 100,000 windows for the train-split statistics, checks the raw-signal loader against the waveform files and rebuilds `case_meta.parquet`) and prints one PASS/FAIL line per requirement. It writes `results/bshi_training_audit.json`, `data/derived/bshi/context_stats_v1.json`, `data/derived/bshi/target_stats_v1.json` and `docs/bshi_training_readiness.md` (verdict and the table requirement / have it / evidence / action). Set `REFRESH = False` to only print the gates of the last saved run.

In [ ]:
from hemodynamics import bshi_audit as AU
REFRESH = True
audit = AU.run_audit(ROOT) if REFRESH else json.loads((ROOT / AU.RESULT).read_text())
ok = AU.print_gates(audit)
if REFRESH:
    print('readiness document lines:', AU.write_readiness_doc(ROOT, audit))
print('TRAINING READINESS: ALL GATES PASS' if ok else 'TRAINING READINESS: A GATE FAILED, send the lines above back')